<a href="https://colab.research.google.com/github/pedromilken/Disciplinas-Doutorado/blob/main/aula0_4_luong_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparação dos dados

Esta tarefa é inverter sequências de caracteres. Exemplo: **aabcd** em **dcbaa**.


In [ ]:
import torch
import torch.nn as nn
import random
import torch.nn.functional as F

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dados
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(50000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Veja um par

In [ ]:
print(pairs[1])

# Definição do modelo Seq2Seq com GRU

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        outputs, h = self.gru(x)
        return outputs, h   # <--- ESSENCIAL

In [ ]:
class LuongAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden: (B, 1, H)
        encoder_outputs: (B, S, H)

        Retorna:
          context: (B, 1, H)
          attn_weights: (B, 1, S)
        """

        # score = h_t · h_s^T
        # (B, 1, H) x (B, H, S) -> (B, 1, S)
        attn_scores = torch.bmm(decoder_hidden, encoder_outputs.transpose(1, 2))

        attn_weights = F.softmax(attn_scores, dim=-1)  # normaliza nos steps da source

        # context = soma ponderada
        # (B, 1, S) x (B, S, H) -> (B, 1, H)
        context = torch.bmm(attn_weights, encoder_outputs)

        return context, attn_weights

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.attn = LuongAttention()

        # Luong concat: concatena hidden + context
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, h, encoder_outputs):
        """
        x: tokens anteriores corretos  (B, T)
        h: estado inicial do decoder   (1, B, H)
        encoder_outputs: todos os h_s  (B, S, H)
        """
        x = self.embed(x)  # (B, T, E)

        outputs = []
        seq_len = x.size(1)
        hidden = h

        for t in range(seq_len):
            inp = x[:, t:t+1]  # (B, 1, E)

            out_t, hidden = self.gru(inp, hidden)   # out_t: (B,1,H)

            # Atenção
            context, attn_w = self.attn(out_t, encoder_outputs)

            # concatenação [out_t ; context]
            combined = torch.cat([out_t, context], dim=-1)

            logits = self.fc(combined)  # (B,1,V)
            outputs.append(logits)

        outputs = torch.cat(outputs, dim=1)  # (B, T, V)
        return outputs, hidden


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        encoder_outputs, h = self.encoder(src)
        logits, _ = self.decoder(tgt[:, :-1], h, encoder_outputs)
        return logits

# Código para usar o modelo treinado: inferência

In [ ]:
def decode_step(decoder, token, h, encoder_outputs):
    """
    Executa um passo de decodificação:
    - token: tensor (B,1)
    - h: estado oculto do decoder (1,B,H)
    - encoder_outputs: (B,S,H)
    """
    logits, h = decoder(token, h, encoder_outputs)  # (B,1,V)
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)  # (B,1)
    return next_token, h


def predict(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        # codifica entrada
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        # encoder agora retorna (encoder_outputs, h)
        encoder_outputs, h = model.encoder(src)

        # token inicial (ex: espaço ou <sos>)
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)

        seq_invertida = []
        for _ in range(max_len):
            token, h = decode_step(model.decoder, token, h, encoder_outputs)
            seq_invertida.append(token.item())

        return decode(seq_invertida)


# Preparação para treino

In [ ]:
emb_size = 32
hidden_size = 64
encoder = Encoder(vocab_size, emb_size, hidden_size)
decoder = Decoder(vocab_size, emb_size, hidden_size)
model = Seq2Seq(encoder, decoder).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Execução do treino

In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt.zero_grad()
        logits = model(xb, yb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()

    # --- Calculate Character Error Rate for the epoch ---
    epoch_char_error_count = 0
    total_chars_compared = 0

    model.eval() # Switch to evaluation mode for inference, disables dropout etc.
    with torch.no_grad(): # Disable gradient calculation for this metric
        for xb_eval, yb_eval in train_dl: # Use the training DataLoader for consistency
            xb_eval = xb_eval.to(device, dtype=torch.long)

            # Get encoder outputs and initial hidden state (h from original GRUEncoder is a tensor)
            encoder_outputs_eval, h_encoder_eval = model.encoder(xb_eval) # h_encoder_eval is (1, B, H)

            # Initial decoder token (e.g., space or <sos>) for the batch
            token_eval = torch.tensor([[vocab[' ']]] * xb_eval.size(0), dtype=torch.long, device=device) # (B, 1)

            predicted_decoded_tokens_batch = []
            current_h_eval = h_encoder_eval # For original GRU-GRU, this is straightforward

            for _ in range(max_len): # max_len is defined globally (5)
                # decode_step for original model needs (decoder, token, h, encoder_outputs)
                # The original Decoder (0eanY-mUa6IU) forward: def forward(self, x, h, encoder_outputs):
                logits_t, current_h_eval = model.decoder(token_eval, current_h_eval, encoder_outputs_eval) # logits_t: (B, 1, V)
                next_token_eval = logits_t.argmax(dim=-1) # (B, 1)
                predicted_decoded_tokens_batch.append(next_token_eval)
                token_eval = next_token_eval # Update token for next step

            # Concatenate to get batch of predicted sequences (B, max_len)
            predicted_decoded_tokens_batch = torch.cat(predicted_decoded_tokens_batch, dim=1)

            # Now, compare original input `xb_eval` with the reversed `predicted_decoded_tokens_batch`
            for j in range(xb_eval.size(0)): # Iterate through each sequence in the current batch
                input_seq_tensor = xb_eval[j] # (max_len,)
                predicted_output_seq_tensor = predicted_decoded_tokens_batch[j] # (max_len,)

                input_str = decode(input_seq_tensor).replace(' ', '')
                predicted_str = decode(predicted_output_seq_tensor).replace(' ', '')

                # Reverse the predicted string for comparison
                reversed_predicted_str = predicted_str[::-1]

                # Compare char by char. max_len is 5, random_seq generates length 5, so all are length 5.
                compare_len = min(len(input_str), len(reversed_predicted_str))

                for k in range(compare_len):
                    if input_str[k] != reversed_predicted_str[k]:
                        epoch_char_error_count += 1
                total_chars_compared += compare_len # Total number of characters effectively compared

    average_char_error = epoch_char_error_count / total_chars_compared if total_chars_compared > 0 else 0
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}, char_error={average_char_error:.4f}")

    model.train() # Switch back to training mode


# Vamos testar

In [ ]:
for _ in range(10):
    s = random_seq()
    pred = predict(model, s, max_len=len(s))
    print(f"{s} -> {pred}")


# Exercício
Compare o resultado do uso do encoder-decoder com atenção com o encoder-decoder sem atenção.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# --- 1. Encode specific sequences and prepare for PCA ---
sequences_to_test = [
    "aaaabb",
    "bbaaab",
    "cbcaccc",
    "cccacbc",
    "abcd ", # A different sequence
    "dcba "  # Its reverse
]

# Collect encoder hidden states
hidden_states = []
labels = []

model.eval()
with torch.no_grad():
    for seq_str in sequences_to_test:
        # Encode and pad the sequence, similar to the predict function
        src = pad(encode(seq_str)).unsqueeze(0).to(device, dtype=torch.long)

        # Get encoder outputs and the final hidden state
        _, h = model.encoder(src)

        # The hidden state `h` is (1, B, H). We need (H,) for PCA on a single sequence.
        hidden_states.append(h.squeeze().cpu().numpy())
        labels.append(seq_str)

# Convert list of hidden states to a NumPy array
hidden_states_np = np.array(hidden_states)

# --- 2. Apply PCA ---
pca = PCA(n_components=2)
pca_components = pca.fit_transform(hidden_states_np)

# --- 3. Plot PCA results ---
plt.figure(figsize=(8, 6))
plt.scatter(pca_components[:, 0], pca_components[:, 1])

for i, txt in enumerate(labels):
    plt.annotate(txt, (pca_components[i, 0], pca_components[i, 1]), textcoords="offset points", xytext=(5,5), ha='center')

plt.title('PCA of Encoder Hidden States')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

print(f"Explained variance ratio by principal components: {pca.explained_variance_ratio_}")

# --- 4. Calculate model accuracy ---
def calculate_accuracy(model, num_samples=100):
    correct_predictions = 0
    total_predictions = 0

    model.eval()
    with torch.no_grad():
        for _ in range(num_samples):
            s = random_seq(n=max_len) # Generate sequences of max_len

            # True reversed sequence (target)
            true_reverse = s[::-1]

            # Model's prediction
            predicted_reverse = predict(model, s, max_len=len(s)).replace(' ', '') # Remove padding spaces from prediction

            if predicted_reverse == true_reverse:
                correct_predictions += 1
            total_predictions += 1

    accuracy = (correct_predictions / total_predictions) * 100
    return accuracy

# Calculate and print accuracy
accuracy = calculate_accuracy(model, num_samples=1000)
print(f"\nModel Accuracy on 1000 random sequences: {accuracy:.2f}%")


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- New Encoder Classes ---
# Renamed to avoid conflicts with previous definitions if present in other cells
class GRUEncoderNew(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        outputs, h = self.gru(x)
        return outputs, h # h is a tensor

class LSTMEncoderNew(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        outputs, (h, c) = self.lstm(x) # LSTM returns (h_n, c_n) tuple
        return outputs, (h, c) # Return the tuple

# --- New Decoder Classes (with Luong Attention) ---
# Assumes LuongAttention class is already defined and available in the global scope.
class GRUDecoderNew(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.attn = LuongAttention() # Use the previously defined LuongAttention
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, h, encoder_outputs):
        x = self.embed(x)
        outputs = []
        seq_len = x.size(1)
        hidden = h # GRU hidden state is a single tensor

        for t in range(seq_len):
            inp = x[:, t:t+1]  # (B, 1, E)
            out_t, hidden = self.gru(inp, hidden)   # out_t: (B,1,H)
            context, attn_w = self.attn(out_t, encoder_outputs)
            combined = torch.cat([out_t, context], dim=-1)
            logits = self.fc(combined)  # (B,1,V)
            outputs.append(logits)

        outputs = torch.cat(outputs, dim=1)  # (B, T, V)
        return outputs, hidden

class LSTMDecoderNew(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(emb_size, hidden_size, batch_first=True)
        self.attn = LuongAttention() # Use the previously defined LuongAttention
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, h, encoder_outputs):
        x = self.embed(x)
        outputs = []
        seq_len = x.size(1)
        hidden = h # LSTM hidden state is a tuple (h_n, c_n)

        for t in range(seq_len):
            inp = x[:, t:t+1]
            out_t, hidden = self.lstm(inp, hidden) # hidden updated to new (h_n, c_n)
            context, attn_w = self.attn(out_t, encoder_outputs)
            combined = torch.cat([out_t, context], dim=-1)
            logits = self.fc(combined)
            outputs.append(logits)

        outputs = torch.cat(outputs, dim=1)
        return outputs, hidden

# --- Generic Seq2Seq Class to combine any Encoder and Decoder ---
class Seq2SeqNew(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        encoder_outputs, h_encoder = self.encoder(src)

        # Adapt h_encoder to decoder's expected hidden state format
        if isinstance(self.decoder, GRUDecoderNew):
            # GRU decoder expects a single tensor hidden state
            if isinstance(h_encoder, tuple): # If encoder is LSTM, take h_n part
                h_decoder = h_encoder[0]
            else: # If encoder is GRU, it's already a single tensor
                h_decoder = h_encoder
        elif isinstance(self.decoder, LSTMDecoderNew):
            # LSTM decoder expects a tuple (h_n, c_n)
            if isinstance(h_encoder, torch.Tensor): # If encoder is GRU, duplicate h for c_n
                h_decoder = (h_encoder, h_encoder)
            else: # If encoder is LSTM, it's already a tuple
                h_decoder = h_encoder
        else:
            raise ValueError("Unsupported decoder type")

        logits, _ = self.decoder(tgt[:, :-1], h_decoder, encoder_outputs)
        return logits

# --- Modified decode_step function to handle both GRU and LSTM hidden states ---
def decode_step_new(decoder, token, h, encoder_outputs):
    logits, h_new = decoder(token, h, encoder_outputs)  # (B,1,V)
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)  # (B,1)
    return next_token, h_new

# --- Modified predict function to use the new classes and decode_step ---
def predict_new(model, seq, max_len):
    model.eval()
    with torch.no_grad():
        # Encode input sequence
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)

        # Get encoder outputs and initial hidden state (can be tensor or tuple)
        encoder_outputs, h = model.encoder(src)

        # Adapt h for the decoder's prediction loop
        if isinstance(model.decoder, GRUDecoderNew):
            # GRU decoder expects a single tensor hidden state
            if isinstance(h, tuple):
                h_initial = h[0]
            else:
                h_initial = h
        elif isinstance(model.decoder, LSTMDecoderNew):
            # LSTM decoder expects a tuple (h_n, c_n)
            if isinstance(h, torch.Tensor):
                h_initial = (h, h)
            else:
                h_initial = h
        else:
            raise ValueError("Unsupported decoder type for predict_new")

        # Initial decoder token (e.g., space or <sos>)
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)

        inverted_seq_tokens = []
        current_h = h_initial
        for _ in range(max_len):
            token, current_h = decode_step_new(model.decoder, token, current_h, encoder_outputs)
            inverted_seq_tokens.append(token.item())

        return decode(inverted_seq_tokens)

# --- Training function ---
def train_model(model_instance, train_dl, loss_fn, opt, epochs=10):
    enc_name = type(model_instance.encoder).__name__.replace('EncoderNew', '')
    dec_name = type(model_instance.decoder).__name__.replace('DecoderNew', '')
    model_combo_name = f"{enc_name}-{dec_name}"
    print(f"Iniciando treinamento para o modelo {model_combo_name}...")
    for epoch in range(epochs):
        model_instance.train()
        total_loss = 0
        for xb, yb in train_dl:
            xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
            opt.zero_grad()
            logits = model_instance(xb, yb)
            # Reshape yb for CrossEntropyLoss, ignoring the first token of target (teacher forcing start token)
            loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
            loss.backward()
            opt.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")
    print(f"Treinamento do modelo {model_combo_name} concluído.\n")

# --- Instantiate and train each model ---
models_to_train = {}
optimizers = {}

# Mapping model names to their respective Encoder and Decoder classes
all_model_names_combinations = {
    "GRU-GRU": (GRUEncoderNew, GRUDecoderNew),
    "GRU-LSTM": (GRUEncoderNew, LSTMDecoderNew),
    "LSTM-GRU": (LSTMEncoderNew, GRUDecoderNew),
    "LSTM-LSTM": (LSTMEncoderNew, LSTMDecoderNew)
}

print("### Treinamento dos 4 Modelos Seq2Seq com Atenção de Luong ###\n")

for model_name, (encoder_class, decoder_class) in all_model_names_combinations.items():
    # Instantiate encoder and decoder for the current model combination
    encoder_instance = encoder_class(vocab_size, emb_size, hidden_size)
    decoder_instance = decoder_class(vocab_size, emb_size, hidden_size)

    # Create the Seq2Seq model and move it to the device
    current_model = Seq2SeqNew(encoder_instance, decoder_instance).to(device)
    # Initialize an optimizer for the current model
    current_optimizer = torch.optim.Adam(current_model.parameters(), lr=1e-3)

    # Store model and optimizer
    models_to_train[model_name] = current_model
    optimizers[model_name] = current_optimizer

    # Train the current model
    train_model(current_model, train_dl, loss_fn, current_optimizer, epochs=10)

# --- Evaluation for each trained model ---
print("### Resultados da Avaliação para os 4 Modelos Seq2Seq com Atenção de Luong ###\n")

evaluation_inputs = [
    "adcdd",
    "adabc",
    "adbbc",
    "dabbd",
    "addcb"
]

# User-provided expected predictions (from the prompt)
expected_predictions = {
    "GRU-GRU": {
        "adcdd": "ddcac", "adabc": "cabdd", "adbbc": "cbbad", "dabbd": "dbbbd", "addcb": "bcdda"
    },
    "GRU-LSTM": {
        "adcdd": "dddab", "adabc": "cbabc", "adbbc": "bcbcc", "dabbd": "bddab", "addcb": "bbcdd"
    },
    "LSTM-GRU": {
        "adcdd": "dddcd", "adabc": "cabab", "adbbc": "cbbad", "dabbd": "dbbaa", "addcb": "bdcac"
    },
    "LSTM-LSTM": {
        "adcdd": "ddcab", "adabc": "cbadc", "adbbc": "cbbda", "dabbd": "bdbad", "addcb": "bcdbd"
    }
}

for model_name, model_instance in models_to_train.items():
    print(f"Predições para o Modelo: {model_name}")
    for inp_seq in evaluation_inputs:
        # Predict using the model and current max_len
        predicted_output = predict_new(model_instance, inp_seq, max_len).replace(' ', '')
        expected_output = expected_predictions[model_name][inp_seq]
        match = "✓ Correto" if predicted_output == expected_output else f"✗ Incorreto (Esperado: {expected_output})"
        print(f"  Entrada: {inp_seq} -> Saída Prevista: {predicted_output} ({match})")
    print("\n")


In [ ]:
def calculate_accuracy_for_new_models(model_instance, num_samples=1000, max_len=max_len):
    correct_predictions = 0
    total_predictions = 0

    model_instance.eval()
    with torch.no_grad():
        for _ in range(num_samples):
            s = random_seq(n=max_len) # Generate sequences of max_len

            # True reversed sequence (target)
            true_reverse = s[::-1]

            # Model's prediction using the new predict_new function
            predicted_reverse = predict_new(model_instance, s, max_len=len(s)).replace(' ', '') # Remove padding spaces from prediction

            if predicted_reverse == true_reverse:
                correct_predictions += 1
            total_predictions += 1

    accuracy = (correct_predictions / total_predictions) * 100
    return accuracy

print("\n### Acurácia dos Modelos Treinados ###\n")
for model_name, model_instance in models_to_train.items():
    accuracy = calculate_accuracy_for_new_models(model_instance, num_samples=10000)
    print(f"Acurácia do Modelo {model_name} em 1000 sequências aleatórias: {accuracy:.2f}%")
